In [1]:
# --- reader / flat-pack maker (robust) ---
import os, csv
import numpy as np

OUT_DIR = "lambda_dump"  # same folder you saved in

def load_Lambda(k_index, layer, sym, out_dir=OUT_DIR):
    fname = os.path.join(out_dir, f"k{k_index:05d}_L{layer}_S{sym}.npz")
    z = np.load(fname, allow_pickle=True)
    return {
        "k": z["k"],
        "layer": int(z["layer"]),
        "sym": int(z["sym"]),
        "Lambda": z["Lambda"],
        "evals_plus": z["evals_plus"],
        "evals_minus": z["evals_minus"],
    }

def _read_k_index_table(in_dir="lambda_dump"):
    path = os.path.join(in_dir, "k_index.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {path}")
    items = []
    with open(path, "r", newline="") as f:
        for row in csv.DictReader(f):
            items.append((int(row["k_index"]), float(row["kx"]), float(row["ky"])))
    return items  # list of (k_index, kx, ky)

def _quantize_pair(kx, ky, decimals=12):
    return (round(kx, decimals), round(ky, decimals))

def _build_partner_map(all_k, decimals=12):
    """
    Build dict: index_of_k  -> index_of_minus_k (if exists), else None
    """
    by_key = { _quantize_pair(kx, ky, decimals): idx for idx, kx, ky in all_k }
    partners = {}
    for idx, kx, ky in all_k:
        key_minus = _quantize_pair(-kx, -ky, decimals)
        partners[idx] = by_key.get(key_minus, None)
    return partners

def _two_flats(E):
    """Return indices of the two eigenvalues closest to zero."""
    E = np.asarray(E, float)
    o = np.argsort(np.abs(E))
    return np.sort(o[:2])

def _gather_L_blocks(k_index, layer, syms, in_dir):
    """Load Λ for given k_index, layer over syms. Returns list of Λ, and a z for eigenvalues."""
    Ls = []
    z_any = None
    for s in syms:
        z = load_Lambda(k_index, layer=layer, sym=s, out_dir=in_dir)
        Ls.append(z["Lambda"])
        if z_any is None:
            z_any = z
    return Ls, z_any


def make_flatpacks_from_store_singlet(
    *, in_dir="lambda_dump", out_dir="lambda_flat_singlet", k_indices=None, syms=(0,), decimals=12
):
    """
    Singlet (even) channel: (Λ(k) + Λ_partner(k)^T)/2 in your pp convention.
    """
    os.makedirs(out_dir, exist_ok=True)
    all_k = _read_k_index_table(in_dir)
    if k_indices is not None:
        keep = set(int(i) for i in k_indices)
        all_k = [t for t in all_k if t[0] in keep]

    partner_of = _build_partner_map(all_k, decimals=decimals)
    syms = tuple(int(s) for s in syms)
    nS = len(syms)

    for k_index, kx, ky in all_k:
        pidx = partner_of[k_index]
        if pidx is None:
            continue

        L0, z0 = _gather_L_blocks(k_index, layer=0, syms=syms, in_dir=in_dir)
        L1, z1 = _gather_L_blocks(k_index, layer=1, syms=syms, in_dir=in_dir)
        L0P, _  = _gather_L_blocks(pidx,    layer=0, syms=syms, in_dir=in_dir)
        L1P, _  = _gather_L_blocks(pidx,    layer=1, syms=syms, in_dir=in_dir)

        Eplus  = np.asarray(z0["evals_plus"],  float)
        Eminus = np.asarray(z0["evals_minus"], float)
        k_vec  = np.asarray(z0["k"], float)
        
        size_matrix,size_matrix=np.asarray(L0[0]).shape


        L0flat = np.empty((nS, size_matrix,size_matrix), dtype=complex)
        L1flat = np.empty((nS, size_matrix, size_matrix), dtype=complex)
        for si, s in enumerate(syms):
            # singlet: symmetric under k→-k (transpose in your pp convention)
            L0flat[si] = (L0[si] + L0P[si].T) / 2.0
            L1flat[si] = (L1[si] + L1P[si].T) / 2.0

        out_path = os.path.join(out_dir, f"flat_k{k_index:05d}.npz")
        np.savez_compressed(
            out_path,
            L0flat=L0flat, L1flat=L1flat,
            Eplus_flat=Eplus,
            Eminus_flat=Eminus,
            k=k_vec, k_index=int(k_index),
            partner_index=int(pidx),
            syms=np.array(syms, int),
        )
    print(f"Flat packs written to {out_dir}/")

make_flatpacks_from_store_singlet(in_dir="lambda_dump", out_dir="lambda_singlet", syms=(0,))

Flat packs written to lambda_singlet/
